# YOLOv8 - Treinamento de modelo customizado para vazamentos de gás

O dataset que criamos para o treinamento desse modelo pode ser encontrado no link a seguir:  https://app.roboflow.com/roberli/detector-de-gas/browse?queryText=&pageSize=50&startingIndex=0&browseQuery=true

Os demais arquivos necessários para a execução desse código podem ser encontrados no seguinte repositório:

https://drive.google.com/drive/folders/1fbmdwq0HorSvJ0A4UIHge39IE8-hVHQb?usp=drive_link


In [ ]:
!nvidia-smi

## Preparação do dataset

A estrutura necessária é a seguinte

* /dataset
  * /train
  * /val

In [ ]:
!mkdir dataset

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

### Copiando o conjunto de imagens de treinamento

In [ ]:
!cp /content/gdrive/MyDrive/GasLeak/obj.zip ./

In [ ]:
!unzip obj.zip -d dataset/train

In [ ]:
!mv dataset/ dataset/train

### Copiando o conjunto de imagens de validação

In [ ]:
!cp /content/gdrive/MyDrive/GasLeak/valid.zip ./

In [ ]:
!unzip valid.zip -d dataset/valid

## Instalação das ferramentas do YOLOv8

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt


## Configurações do arquivo YAML

In [ ]:
!touch configs_modelo.yaml

Agora, precisamos preencher esse arquivo com os parâmetros necessários

* path: [caminho do diretório que contém o dataset]
* train: [caminho do conjunto de treinamento, relativo ao path]
* val: [caminho do conjunto de validação, relativo ao path]
* test: [não é necessário pois faremos depois um teste manual]

**Número de Classes**
* nc: [coloca o numero de classes que deseja treinar]

**Nomes/Labels - substitua pelos nomes das classes**
* names: [nome de cada classe, dentro de '' e separado por vírgula]

Para escrever esses valores no arquivo, podemos usar o comando %%writefile

In [ ]:
%%writefile configs_modelo.yaml
path: '/content/dataset/'
train: 'train/'
val: 'valid/'
test: # opcional

nc: 1
names: ['gasleak']

## Treinamento do modelo

In [ ]:
diretorio_raiz = '/content/'
arquivo_config = os.path.join(diretorio_raiz, 'configs_modelo.yaml')

In [ ]:
arquivo_config

In [ ]:
#!yolo task=detect mode=train data={arquivo_config} epochs=10

> **Parâmetros da função de treinamento:**

* task: o que queremos com o treinamento. Como estamos trabalhando com detecção de objetos, deixe o valor = detect. Outras opções aceitas: segment, e classify. É opcional passarmos se queremos detecção, pois por padrão ele já considera como sendo detecção a não ser que especifique outra forma.
* mode: pode ser train, val, ou predict. Como estamos fazendo pela forma usando python e queremos o treinamento vamos usar a função train(), portanto esse parâmetro se torna desnecessário .
* **model**: o modelo pré-treinado que queremos usar como "partida". Pode ser o YOLOv8 Nano (YOLOv8n), YOLOv8 Small (YOLOv8s), etc.
* **imgsz**: O tamanho da imagem, que a rede realiza o processamento (obs: você não precisa redimensionar a imagem para esse tamanho antes, o algoritmo cuida dessa parte antes de passar a imagem de entrada para a rede). A resolução padrão é 640x640 pixels, portanto o valor padrão é 640. Quando maior o tamanho mais precisa é a detecção, principalmente para objetos com detalhes pequenos, porém é mais demorado o treinamento e detecção.  
* **data**: caminho para o arquivo YAML. Esse é o arquivo que criamos acima, que contém o caminho para o conjunto de treinamento e validaçao, além disso deve conter os nomes das classes que queremos treinar.
* **epochs**: Numero de epocas que desejamos treinar.
* **batch**: O tamanho do batch (lote) para o carregador de dados. Você pode aumentá-lo ou diminuí-lo de acordo com a disponibilidade de memória de sua GPU, por exemplo caso venha a encontrar problema de memória. O valor padrão é 16.
* **name**: Nome do diretório de resultados para o runs/detect. (opcional)

Em nossos testes vamos escolher o Nano, e 500 épocas, pois queremos que seja um treinamento relativamente mais rápido

In [ ]:
model = YOLO('yolov8s.yaml')

In [ ]:
resultados = model.train(data=arquivo_config, epochs=500, imgsz=640, name='yolov8s_modelo')

In [ ]:
dir_resultado = '/content/gdrive/MyDrive/GasLeak/yolov8s_modelo'

### Avaliação (Verificando o mAP do modelo)

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
!yolo task=detect mode=val model={dir_resultado}/weights/best.pt name=yolov8s_modelo_eval data=configs_modelo.yaml

### Exibindo os gráficos

In [ ]:
def mostrar(img):
  fig = plt.gcf()
  fig.set_size_inches(16, 10)
  plt.axis("off")
  plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
  plt.show()

In [ ]:
#resultados_grafico

In [ ]:
resultados_grafico = cv2.imread(os.path.join(dir_resultado, 'results.png'))
mostrar(resultados_grafico)

In [ ]:
dir_resultado_val = 'runs/detect/yolov8s_modelo_eval'

In [ ]:
imgs = ['F1_curve.png', 'PR_curve.png', 'P_curve.png', 'R_curve.png']
plt.figure(figsize=(18,14))
for i, img in enumerate(imgs):
  #print(i, img)
  grafico = cv2.imread(os.path.join(dir_resultado_val, img))
  #print(grafico)
  grafico = cv2.cvtColor(grafico, cv2.COLOR_BGR2RGB)
  plt.subplot(2, 2, i + 1)
  plt.title(imgs[i])
  plt.imshow(grafico)
  plt.axis('off')
plt.show()

In [ ]:
matriz_confusao = cv2.imread(os.path.join(dir_resultado_val, 'confusion_matrix.png'))
mostrar(matriz_confusao)

## Testando o modelo treinado


In [ ]:
!mkdir imagens_teste

In [ ]:
!cp /content/gdrive/MyDrive/GasLeak/best.pt /content/imagens_teste

In [ ]:
ls

In [ ]:
cd /content/gdrive/MyDrive/GasLeak/Imagens_Teste

In [ ]:
!cp -r /content/gdrive/MyDrive/GasLeak/Imagens_Teste /content/imagens_teste

In [ ]:
!yolo task=detect mode=predict model={dir_resultado}/weights/best.pt source='/content/imagens_teste/Imagens_Teste' save=true conf=0.5

In [ ]:
dir_predicoes = 'runs/detect/predict3'
caminhos = [os.path.join(dir_predicoes, f) for f in os.listdir(dir_predicoes)]
#print(caminhos)
for caminho_imagem in caminhos:
  imagem = cv2.imread(caminho_imagem)
  mostrar(imagem)

## Continuar treinamento



In [ ]:
#!yolo task=detect mode=train model={dir_resultado}/weights/last.pt data={arquivo_config} epochs=(definir número de épocas)

## Enviar para o Google Drive

In [ ]:
!cp -R {dir_resultado} /content/gdrive/MyDrive/GasLeak

## Exportando para outros formatos

In [ ]:
os.path.join(dir_resultado, 'weights', 'best.pt')

In [ ]:
model_treinado = YOLO(os.path.join(dir_resultado, 'weights', 'best.pt'))
model_treinado.export(format='onnx')

Na tabela abaixo, destacamos os formatos aceitos e qual o valor do parâmetro deve ser usado para salvar no formato específico

| Format                                                                     | `format=`          | Model                     |
|----------------------------------------------------------------------------|--------------------|---------------------------|
| [PyTorch](https://pytorch.org/)                                            | -                  | `yolov8n.pt`              |
| [TorchScript](https://pytorch.org/docs/stable/jit.html)                    | `torchscript`      | `yolov8n.torchscript`     |
| [ONNX](https://onnx.ai/)                                                   | `onnx`             | `yolov8n.onnx`            |
| [OpenVINO](https://docs.openvino.ai/latest/index.html)                     | `openvino`         | `yolov8n_openvino_model/` |
| [TensorRT](https://developer.nvidia.com/tensorrt)                          | `engine`           | `yolov8n.engine`          |
| [CoreML](https://github.com/apple/coremltools)                             | `coreml`           | `yolov8n.mlmodel`         |
| [TensorFlow SavedModel](https://www.tensorflow.org/guide/saved_model)      | `saved_model`      | `yolov8n_saved_model/`    |
| [TensorFlow GraphDef](https://www.tensorflow.org/api_docs/python/tf/Graph) | `pb`               | `yolov8n.pb`              |
| [TensorFlow Lite](https://www.tensorflow.org/lite)                         | `tflite`           | `yolov8n.tflite`          |
| [TensorFlow Edge TPU](https://coral.ai/docs/edgetpu/models-intro/)         | `edgetpu`          | `yolov8n_edgetpu.tflite`  |
| [TensorFlow.js](https://www.tensorflow.org/js)                             | `tfjs`             | `yolov8n_web_model/`      |
| [PaddlePaddle](https://github.com/PaddlePaddle)                            | `paddle`           | `yolov8n_paddle_model/`   |





## Detectando objetos em videos

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
cd /content/

In [ ]:
!mkdir videos_teste

### Carregando o vídeo do drive

In [ ]:
ls

In [ ]:
cd /content/gdrive/MyDrive/GasLeak/Videos_teste

In [ ]:
!cp -r /content/gdrive/MyDrive/GasLeak/Videos_teste /content/dataset/videos_teste

In [ ]:
!yolo task=detect mode=predict model={dir_resultado}/weights/best.pt source='/content/gdrive/MyDrive/GasLeak/Videos_teste' save=true conf=0.4

In [ ]:
import os
video_salvo = '/content/runs/detect/predict2/Vídeo_teste_vazamento_de_gás_1.avi'
video_final = '/content/runs/detect/predict2/Vídeo_teste_vazamento_de_gás_1_resultado.mp4'
os.system(f"ffmpeg -i {video_salvo} -vcodec libx264 {video_final}") #renomear os videos retirando os espaços e parênteses

In [ ]:
def exibir_video(video_caminho):
  from IPython.display import HTML
  from base64 import b64encode
  mp4 = open(video_caminho, 'rb').read()
  data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
  return HTML("""
  <video width=500 controls>
    <source src="%s" type="video/mp4"
  </video>
  """ % data_url)

In [ ]:
exibir_video(video_final)